# DX 704 Week 12 Project

This week's project will revisit the email spam classifier project from week 9 using large language model embeddings instead of custom features.


The full project description and a template notebook are available on GitHub: [Project 12 Materials](https://github.com/bu-cds-dx704/dx704-project-12).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

--2026-11-18 18:22:07--  https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip [following]
--2026-11-18 18:22:07--  https://raw.githubusercontent.com/MWiechmann/enron_spam_data/refs/heads/master/enron_spam_data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15642124 (15M) [application/zip]
Saving to: ‘enron_spam_data.zip’

enron_spam_data.zip 100%[===================>]  14.92M  42.0MB/s    in 0.4s    

2026-11-18 18:22:08 (42.0 MB/s) - ‘enron_spa

In [22]:
import pandas as pd
import json

In [2]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [3]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Download BERT Model

We will use a pre-trained BERT model to extract embedding vectors as described in lesson 2.1 this week.
Here is sample code to download the model from [Hugging Face](https://huggingface.co/) and extract one vector.
This model is small enough that you can run it with CPU only, but GPUs will be faster if available.

In [ ]:
# You may need to install torch and transformers.
# Google Colab will have these installed already.
#
# pip install transformers torch --upgrade

import torch
from transformers import AutoTokenizer, AutoModel

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [6]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.to(device)
bert_model.eval()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [7]:
@torch.no_grad()
def embed_text(text):
    batch = [text]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    outputs = bert_model(**inputs)
    # CLS token embedding is the first token's hidden state
    cls_emb = outputs.last_hidden_state[:, 0, :]  # shape: [batch_size, 768]
    return cls_emb.cpu()

In [9]:
v = embed_text("Hi, will you buy my spam?")
v.shape

torch.Size([1, 768])

## Part 3: Create Embedding Vectors

Use BERT to create embeddings for each email in the Enron data set.
You will have to decide how to combine the different columns of the original data set to produce one embedding vector.


Hint: BERT can be run without a GPU, but will be much slower.
Using Google Colab with only a CPU, it runs around 1 embedding per second.
Using Google Colab with the T4 GPU option, it runs around 60 embeddings per second.
Caching is also encouraged to avoid unnecessary reruns.

In [11]:
enron_spam_data['Subject'].isna().any()

np.True_

In [12]:
enron_spam_data['Message'].isna().any()

np.True_

In [17]:
def combine_email_subject_message(row):
    subject = "" if pd.isna(row["Subject"]) else str(row["Subject"])
    message = "" if pd.isna(row["Message"]) else str(row["Message"])
    return f"{subject}\n{message}"

In [19]:
enron_spam_data['embedding_vector'] = enron_spam_data.apply(
    lambda row: embed_text(combine_email_subject_message(row))[0].tolist(),
    axis=1
)
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date,embedding_vector
0,0,christmas tree farm pictures,NaN,ham,1999-12-10,"[-0.4015001654624939, 0.294897198677063, -0.10..."
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13,"[-0.6057171821594238, -0.049807123839855194, 0..."
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14,"[-0.7509446144104004, -0.09142068773508072, -0..."
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14,"[-0.7711719870567322, -0.2957014739513397, -0...."
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14,"[-0.7081449031829834, -0.18517383933067322, 0...."
...,...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29,"[-0.049681682139635086, -0.10663871467113495, ..."
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29,"[0.09933286905288696, -0.02750219963490963, 0...."
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30,"[-0.009408943355083466, -0.15310227870941162, ..."
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30,"[0.23140867054462433, -0.02288445271551609, 0...."


In [24]:
embeddings_df = pd.DataFrame({
    'Message ID': enron_spam_data['Message ID'],
    'embedding_vector_json': (
        enron_spam_data["embedding_vector"].apply(
            lambda v: json.dumps([round(float(x), 3) for x in v])
        )
    )
}
)
embeddings_df

,Message ID,embedding_vector_json
0,0,"[-0.402, 0.295, -0.107, -0.142, 0.291, -0.023,..."
1,1,"[-0.606, -0.05, 0.192, 0.113, 0.343, -0.239, -..."
2,2,"[-0.751, -0.091, -0.291, -0.252, -0.028, -0.44..."
3,3,"[-0.771, -0.296, -0.242, -0.205, -0.422, -0.21..."
4,4,"[-0.708, -0.185, 0.29, -0.217, -0.485, 0.03, 0..."
...,...,...
33711,33711,"[-0.05, -0.107, 0.34, -0.129, -0.299, -0.564, ..."
33712,33712,"[0.099, -0.028, 0.346, 0.085, 0.039, -0.412, 0..."
33713,33713,"[-0.009, -0.153, 0.101, -0.149, -0.393, -0.17,..."
33714,33714,"[0.231, -0.023, 0.043, -0.475, -0.157, -0.668,..."


Save your embeddings in a file "embeddings.tsv.gz" with two columns, Message ID and embedding_vector_json, where embedding_vector_json is a JSON-encoded list.
Make sure that embedding_vector_json is a 1 dimensional list, not 2 dimensional.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

Hint: Gradescope only allows files up to 100MB to be submitted.
Round your embeddings to 3 decimal places to make them smaller.

In [25]:
embeddings_df.to_csv('embeddings.tsv.gz', sep="\t", index=False)

Submit "embeddings.tsv.gz" in Gradescope.

## Part 4: Train a Linear Regression

Train an ordinary least squares regression for spam/ham status where spam is treated as target value 1 and ham is treated as target value 0 with your embeddings above as the only input variables.


In [26]:
from sklearn.linear_model import LinearRegression


model = LinearRegression()

In [38]:
import numpy as np


X = np.vstack(embeddings_df["embedding_vector_json"].map(json.loads).to_numpy())
X

array([[-0.402,  0.295, -0.107, ..., -0.235,  0.305,  0.08 ],
       [-0.606, -0.05 ,  0.192, ..., -0.185,  0.269,  0.382],
       [-0.751, -0.091, -0.291, ..., -0.011,  0.582,  0.513],
       ...,
       [-0.009, -0.153,  0.101, ..., -0.277,  0.303,  0.488],
       [ 0.231, -0.023,  0.043, ..., -0.567,  0.285,  0.33 ],
       [ 0.016, -0.22 ,  0.248, ..., -0.439,  0.331,  0.627]],
      shape=(33716, 768))

In [34]:
y = (enron_spam_data['Spam/Ham'] == 'spam').astype(int)
y

0        0
1        0
2        0
3        0
4        0
        ..
33711    1
33712    1
33713    1
33714    1
33715    1
Name: Spam/Ham, Length: 33716, dtype: int64

In [39]:
model.fit(X, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


Save the coefficients of your linear model in a file "coefficients.tsv" with columns dim and coefficient where dim specifies the offset in the embedding vector (0-767).
Don't worry about the bias term (but your model should still have it).

In [44]:
coef_df = pd.DataFrame({
    'dim': np.arange(model.coef_.size),
    'coefficient': model.coef_
})
coef_df

,dim,coefficient
0,0,0.073331
1,1,0.031331
2,2,0.018823
3,3,0.011855
4,4,0.057969
...,...,...
763,763,0.027293
764,764,0.035100
765,765,0.075845
766,766,-0.078194


In [45]:
coef_df.to_csv('coefficients.tsv', sep='\t', index=False)

Submit "coefficients.tsv" in Gradescope.

## Part 5: Search for Relevant Documents

The file "queries.tsv" specifies 10 queries.
For each of the queries, encode them as a vector, and find the message that is closest using $L_2$.

In [46]:
queries_df = pd.read_csv('queries.tsv', sep='\t')
queries_df

,query_id,query
0,1,accounting arrangements
1,2,sales higher than production
2,3,asked lawyer to write letter about unexpected ...
3,4,engineering exam
4,5,discounted tickets
5,6,unexecuted agreement
6,7,well bore
7,8,capacity problem
8,9,london partner
9,10,dormant account


In [58]:
X

array([[-0.402,  0.295, -0.107, ..., -0.235,  0.305,  0.08 ],
       [-0.606, -0.05 ,  0.192, ..., -0.185,  0.269,  0.382],
       [-0.751, -0.091, -0.291, ..., -0.011,  0.582,  0.513],
       ...,
       [-0.009, -0.153,  0.101, ..., -0.277,  0.303,  0.488],
       [ 0.231, -0.023,  0.043, ..., -0.567,  0.285,  0.33 ],
       [ 0.016, -0.22 ,  0.248, ..., -0.439,  0.331,  0.627]],
      shape=(33716, 768))

In [65]:
# Use same embed function for each query
query_vectors = np.vstack(
    queries_df["query"]
    .apply(lambda q: embed_text(q).tolist())
    .to_numpy()
)
query_vectors

array([[-0.06660818,  0.48602155, -0.69746459, ..., -0.19223221,
         0.08136658,  0.45735845],
       [-0.8651908 , -0.24639829,  0.21123745, ..., -0.6215905 ,
         0.07633926,  0.2763713 ],
       [-0.29763612, -0.2027519 , -0.19771658, ..., -0.1755634 ,
        -0.11157282,  0.09748789],
       ...,
       [-0.25805306,  0.0560736 , -0.19525211, ..., -0.27709702,
        -0.17752071,  0.2306457 ],
       [-0.32044676,  0.15809201, -0.18441832, ..., -0.2878176 ,
         0.26361087,  0.18701747],
       [-0.29751468, -0.30622262, -0.28148147, ..., -0.304474  ,
         0.05795103,  0.58939737]], shape=(10, 768))

In [ ]:
# Find the email with the smallest L2 distance
closest_rows = [
    ((X - query_vector) ** 2).sum(axis=1).argmin()
    for query_vector in query_vectors
]
closest_rows

[np.int64(3273),
 np.int64(2663),
 np.int64(5057),
 np.int64(13222),
 np.int64(3743),
 np.int64(21810),
 np.int64(18137),
 np.int64(14635),
 np.int64(14635),
 np.int64(15831)]

In [68]:
query_matches = pd.DataFrame({
    "query_id": queries_df["query_id"].to_numpy(),
    "query_vector_json": [
        json.dumps([round(float(x), 3) for x in vector])
        for vector in query_vectors
    ],
    "Message ID": embeddings_df.iloc[closest_rows]["Message ID"].to_numpy(),
})
query_matches

,query_id,query_vector_json,Message ID
0,1,"[-0.067, 0.486, -0.697, 0.012, 0.353, 0.001, -...",3273
1,2,"[-0.865, -0.246, 0.211, -0.16, -0.095, 0.163, ...",2663
2,3,"[-0.298, -0.203, -0.198, 0.104, -0.608, -0.122...",5057
3,4,"[-0.215, 0.199, -0.022, 0.005, 0.379, -0.211, ...",13222
4,5,"[-0.196, -0.049, -0.286, 0.084, 0.672, -0.281,...",3743
5,6,"[-0.542, -0.275, -0.15, -0.028, 0.283, -0.127,...",21810
6,7,"[-0.124, 0.354, 0.073, -0.094, -0.022, -0.217,...",18137
7,8,"[-0.258, 0.056, -0.195, 0.072, -0.175, 0.306, ...",14635
8,9,"[-0.32, 0.158, -0.184, -0.192, 0.056, 0.259, 0...",14635
9,10,"[-0.298, -0.306, -0.281, -0.208, 0.501, 0.198,...",15831


Save your results in a file "query-matches.tsv" with columns query_id, query_vector_json, and Message ID.

In [69]:
query_matches.to_csv('query-matches.tsv', sep='\t', index=False)

Submit "query-matches.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [70]:
None